<a href="https://colab.research.google.com/github/Pepi134/TeamoAIChallange/blob/main/TeamoAI_challange.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Teamo AI - challange!





In [5]:
!pip install flask flask-ngrok

In [ ]:
from flask import Flask, request, jsonify, g
from sentence_transformers import SentenceTransformer, util

# Initialize the Flask application
app = Flask(__name__)

# Load the pre-trained Sentence-BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight and fast SBERT model

# Default list of skills managed by administrators
default_skills = ['Python', 'relational database', 'Software engineering',
                  'data science', 'NLP', 'natural language processing']

# Helper function to initialize the skills list in the request context
def init_skills():
    """
    Initializes the skills list if it is not already set.
    """
    if not hasattr(g, 'admin_skills'):
        g.admin_skills = default_skills.copy()

# Route for users to submit a skill name for matching using SBERT
@app.route('/match_skill_sbert', methods=['POST'])
def match_skill_sbert():
    init_skills()  # Initialize the skills list

    user_data = request.json
    user_skill = user_data.get('skill_name', '')

    if not user_skill:
        return jsonify({'error': 'Skill name is required'}), 400

    # Encode both the user skill and admin skills into embeddings
    admin_embeddings = model.encode(g.admin_skills, convert_to_tensor=True)
    user_embedding = model.encode(user_skill, convert_to_tensor=True)

    # Compute cosine similarities between the user skill and admin skills
    cosine_scores = util.cos_sim(user_embedding, admin_embeddings)[0]

    # Create a list of matches with their similarity scores
    matches = [
        {'skill': g.admin_skills[i], 'score': round(float(cosine_scores[i]) * 100, 2)}
        for i in range(len(g.admin_skills))
    ]

    # Sort matches by score in descending order and return the top five
    matches.sort(key=lambda x: x['score'], reverse=True)

    return jsonify({'matches': matches[:5]}), 200

# Route for administrators to add a new skill to the list
@app.route('/admin/add_skill', methods=['POST'])
def add_skill():
    init_skills()  # Initialize the skills list

    admin_data = request.json
    new_skill = admin_data.get('skill_name', '')

    if not new_skill:
        return jsonify({'error': 'Skill name is required'}), 400

    if new_skill.lower() in [skill.lower() for skill in g.admin_skills]:
        return jsonify({'message': 'Skill already exists in the list'}), 400

    g.admin_skills.append(new_skill)
    return jsonify({'message': f'Skill "{new_skill}" added successfully'}), 201

# Route for administrators to delete an existing skill from the list
@app.route('/admin/delete_skill', methods=['DELETE'])
def delete_skill():
    init_skills()  # Initialize the skills list

    admin_data = request.json
    skill_to_delete = admin_data.get('skill_name', '')

    if not skill_to_delete:
        return jsonify({'error': 'Skill name is required'}), 400

    if skill_to_delete.lower() not in [skill.lower() for skill in g.admin_skills]:
        return jsonify({'message': f'Skill "{skill_to_delete}" not found'}), 404

    # Remove the skill (case-insensitive match)
    g.admin_skills = [skill for skill in g.admin_skills if skill.lower() != skill_to_delete.lower()]

    return jsonify({'message': f'Skill "{skill_to_delete}" deleted successfully'}), 200

# Route for administrators to view all skills
@app.route('/admin/view_skills', methods=['GET'])
def view_skills():
    init_skills()  # Initialize the skills list
    return jsonify({'skills': sorted(g.admin_skills)}), 200

# Run the Flask application
if __name__ == '__main__':
   app.run(debug=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat
